# Squads/Rosters

In [1]:
import pandas as pd

import os
from dotenv import load_dotenv

load_dotenv()

FJELSTUL_DIR = os.getenv("FJELSTUL_DIR")

squads = pd.read_csv(os.path.join(FJELSTUL_DIR, "squads.csv"))
players = pd.read_csv(os.path.join(FJELSTUL_DIR, "players.csv"))
goals = pd.read_csv(os.path.join(FJELSTUL_DIR, "goals.csv"))
bookings = pd.read_csv(os.path.join(FJELSTUL_DIR, "bookings.csv"))
appearances = pd.read_csv(os.path.join(FJELSTUL_DIR, "player_appearances.csv"))
substitutions = pd.read_csv(os.path.join(FJELSTUL_DIR, "substitutions.csv"))
award_winners = pd.read_csv(os.path.join(FJELSTUL_DIR, "award_winners.csv"))

# Aggregate goals per player per tournament
goals_agg = (
    goals.groupby(["player_id", "tournament_id"])
    .agg(
        goals=("goal_id", "count"),
        own_goals=("own_goal", "sum"),
        penalty_goals=("penalty", "sum"),
    )
    .reset_index()
)

# Aggregate appearances (matches played, starts, sub appearances)
apps_agg = (
    appearances.groupby(["player_id", "tournament_id"])
    .agg(
        matches_played=("match_id", "count"),
        starts=("starter", "sum"),
        sub_appearances=("substitute", "sum"),
    )
    .reset_index()
)

# Aggregate bookings
bookings_agg = (
    bookings.groupby(["player_id", "tournament_id"])
    .agg(
        yellow_cards=("yellow_card", "sum"),
        red_cards=("red_card", "sum"),
        second_yellows=("second_yellow_card", "sum"),
    )
    .reset_index()
)

# Aggregate substitutions (times subbed off)
subs_off = (
    substitutions[substitutions["going_off"] == 1]
    .groupby(["player_id", "tournament_id"])
    .agg(times_subbed_off=("substitution_id", "count"))
    .reset_index()
)

# Awards
awards_agg = (
    award_winners.groupby(["player_id", "tournament_id"])["award_name"]
    .apply(lambda x: ", ".join(x))
    .reset_index()
    .rename(columns={"award_name": "awards_won"})
)

# Base join: squads + player bio
df = squads.merge(
    players[["player_id", "birth_date", "player_wikipedia_link"]],
    on="player_id",
    how="left",
)

# Join all metrics
df = (
    df.merge(apps_agg, on=["player_id", "tournament_id"], how="left")
    .merge(goals_agg, on=["player_id", "tournament_id"], how="left")
    .merge(bookings_agg, on=["player_id", "tournament_id"], how="left")
    .merge(subs_off, on=["player_id", "tournament_id"], how="left")
    .merge(awards_agg, on=["player_id", "tournament_id"], how="left")
)

# Fill NaN counts with 0
count_cols = [
    "matches_played",
    "starts",
    "sub_appearances",
    "goals",
    "own_goals",
    "penalty_goals",
    "yellow_cards",
    "red_cards",
    "second_yellows",
    "times_subbed_off",
]
df[count_cols] = df[count_cols].fillna(0).astype(int)

# Filter 2006 onwards
df = df[df["tournament_id"] >= "WC-2006"].reset_index(drop=True)

print(df.shape)
df.head()

(5550, 25)


,key_id,tournament_id,tournament_name,team_id,team_name,team_code,player_id,family_name,given_name,shirt_number,...,starts,sub_appearances,goals,own_goals,penalty_goals,yellow_cards,red_cards,second_yellows,times_subbed_off,awards_won
0,8294,WC-2006,2006 FIFA Men's World Cup,T-02,Angola,AGO,P-24930,Ricardo,João,1,...,3,0,0,0,0,1,0,0,0,NaN
1,8295,WC-2006,2006 FIFA Men's World Cup,T-02,Angola,AGO,P-66972,Airosa,Marco,2,...,0,0,0,0,0,0,0,0,0,NaN
2,8296,WC-2006,2006 FIFA Men's World Cup,T-02,Angola,AGO,P-61845,Jamba,not applicable,3,...,3,0,0,0,0,1,0,0,0,NaN
3,8297,WC-2006,2006 FIFA Men's World Cup,T-02,Angola,AGO,P-06457,Lebo,Lebo,4,...,0,0,0,0,0,0,0,0,0,NaN
4,8298,WC-2006,2006 FIFA Men's World Cup,T-02,Angola,AGO,P-44205,Kali,not applicable,5,...,3,0,0,0,0,0,0,0,0,NaN


In [2]:
# df[df["family_name"].str.contains("Messi", case=False, na=False)]

In [3]:
df.to_csv("player_rosters.csv", index=False)